# 1.RLHF (Reinforcement Learning from Human Feedback)


![RLHF](img/RLHF.png)

## 基本流程和注意点
- pre-train预训练模型
  - 训练通用型知识
  - 产生 $\theta$ 基座(based)参数

- SFT(监督微调)
  - 数据：prompt 通常为高质量的指令/对话
  - 模型：based model
  - loss：标准自回归交叉熵
  - output：初始策略 $\pi_{SFT}$ reference和模型 $\pi_{ref}$
    - SFT模型为后续的reward和ppo提供训练起点

- reward method
  - 使用 $\pi_{SFT}$ 对同一prompt采样回答，人工标注数据集
  - 使用格式 (promt, chosen, rejected)
  - $ loss = \log sigmoid(chosen - rejected)$
  - RM通常基于SFT构建，输出层替换为一个用于输出标量的**奖励值**的回归头(Batch_size, Hidden_size)
    - 该模型在后续的PPO训练中参数固定，为策略模型的生成内容提供奖励信号

- PPO
  - 利用reward训练好的奖励信号，通过RL进一步优化SFT模型，生成更符合人类偏好/奖励更高的内容
  - 涉及到四个模型的协同工作。其基本思想是在最大化奖励模型给出的奖励信号同时，通过KL散度惩罚等措施约束优化后的模型不要过于偏离原始的SFT模型，以保持其原有的语言能力和知识。
![PPO训练架构](img/PPO%20training.png)\
[ppo](img/PPO%20training.png)

## **optimiztion**
- 计算最终奖励：最终的奖励信号并非是RM直接给出的奖励。为了约束策略模型不要偏离太远，会从 `reward_score` 中减去一个与引用模型输出概率分布的KL散度惩罚项：`final_reward = reward_score - beta * kl_divergence`

- 计算优势函数（Advantage）：利用**价值模型**（Critic） 的预测值和实际得到的**最终奖励**，通过**广义优势估计**（GAE） 等方法计算优势函数` advantage`，它衡量了某个动作相对于平均水平的优势程度

## **loss**
* Actor Loss：基于优势函数和重要性采样，使用**PPO-clip**技巧更新**策略模型**的参数，目标是最大化期望奖励，同时限制每次更新的幅度，保证稳定性

* Critic Loss：通常采用均方误差（MSE）损失，让**价值模型**的预测值更接近实际回报（Return），从而更好地估计状态价值
  - $loss_{critic} = \mathrm{MES} (V(s), A(s, a) + V(s))$ 衡量实际预测V，和回报的目标价值。



# 2.Training Reward


In [1]:
# 安装依赖
!pip install datasets transformers peft trl

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple


In [2]:
# 使用windows的bnb
!pip install bitsandbytes

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple


In [3]:
import torch
from datasets import Dataset
import json

C:\Users\hhm18\miniconda3\envs\env_DRL\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


本地已加载模型扫描
```bash
huggingface-cli scan-cache
```

In [4]:
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoConfig, AutoTokenizer, BitsAndBytesConfig, AutoModelForSequenceClassification
from trl import RewardConfig, RewardTrainer, AutoModelForCausalLMWithValueHead

In [5]:
import transformers
import bitsandbytes as bnb 
print("Transformers version:", transformers.__version__)
print("bitsandbytes version:", bnb.__version__)

Transformers version: 4.56.0
bitsandbytes version: 0.47.0


In [6]:
import os
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

In [7]:

# model_name = "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B"
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

# 加载模型和分词器
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,                    # 需要pip bnb
    bnb_4bit_use_double_quant=True,       
    bnb_4bit_compute_dtype=torch.float16,
)
tokenizer = AutoTokenizer.from_pretrained(model_name, padding=True, return_tensors="pt")
# tokenizer.padding_side = "right"
# tokenizer.pad_token = tokenizer.eos_token  #Huggingface可以自动对齐

In [8]:
tokenizer.special_tokens_map

{'eos_token': '<|im_end|>',
 'pad_token': '<|endoftext|>',
 'additional_special_tokens': ['<|im_start|>',
  '<|im_end|>',
  '<|object_ref_start|>',
  '<|object_ref_end|>',
  '<|box_start|>',
  '<|box_end|>',
  '<|quad_start|>',
  '<|quad_end|>',
  '<|vision_start|>',
  '<|vision_end|>',
  '<|vision_pad|>',
  '<|image_pad|>',
  '<|video_pad|>']}

注意这里我们使用了`AutoModelForCausalLMWithValueHead`

对于不同模型参考[Hugging Face](https://huggingface.co/docs/trl/en/models?utm_source=chatgpt.com) , [TRL](https://github.com/huggingface/trl?utm_source=chatgpt.com)
 
我们也可以如下定义一个 `AutoModelForCausalLM + Reward Head`

```python
import torch
from torch import nn
from transformers import AutoModelForCausalLM

class RewardModel(nn.Module):
    def __init__(self, model_name, bnb_config):
        super().__init__()
        # 基座模型：CausalLM
        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            quantization_config=bnb_config
        )
        hidden_size = self.model.config.hidden_size
        
        # reward head（线性层，输出一个分数）
        self.score = nn.Linear(hidden_size, 1, bias=False)

    def forward(self, input_ids=None, attention_mask=None):
        outputs = self.model(input_ids=input_ids, attention_mask=attention_mask, output_hidden_states=True)
        
        # 取最后一个 token 的 hidden state
        hidden_states = outputs.hidden_states[-1]
        last_hidden = hidden_states[:, -1, :]   # [batch, seq_len, hidden_dim]
        
        # 映射成 reward
        reward = self.score(last_hidden)
        return reward

```

In [9]:
# 先加载基座模型/reward model
cfg = AutoConfig.from_pretrained(model_name, trust_remote_code=True)
cfg.num_labels = 1
model = AutoModelForSequenceClassification.from_pretrained(model_name, 
                                            config=cfg,
                                            quantization_config=bnb_config,
                                            device_map="auto",
                                            )

# 对齐 pad token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.pad_token_id


Some weights of Qwen2ForSequenceClassification were not initialized from the model checkpoint at Qwen/Qwen2.5-0.5B-Instruct and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [10]:
print(model)

Qwen2ForSequenceClassification(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear4bit(in_features=896, out_features=896, bias=True)
          (k_proj): Linear4bit(in_features=896, out_features=128, bias=True)
          (v_proj): Linear4bit(in_features=896, out_features=128, bias=True)
          (o_proj): Linear4bit(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear4bit(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear4bit(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear4bit(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), ep

In [11]:
# 检查模型的偏好打分
inputs = tokenizer("测试一下 reward model", return_tensors="pt").to("cuda")
outputs = model(**inputs)
print(outputs)                # 应该是 SequenceClassifierOutput
print(outputs.logits.shape)   # torch.Size([batch_size, 1]

C:\Users\hhm18\miniconda3\envs\env_DRL\lib\site-packages\transformers\integrations\sdpa_attention.py:83: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:455.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(


SequenceClassifierOutputWithPast(loss=None, logits=tensor([[-1.9297]], device='cuda:0', dtype=torch.float16,
       grad_fn=<IndexBackward0>), past_key_values=DynamicCache(layers=[DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer]), hidden_states=None, attentions=None)
torch.Size([1, 1])


In [12]:
model.config.num_labels

1

In [13]:
# 设置lora超参，训练奖励模型
peft_config = LoraConfig(
    r = 4,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",             # 对投影层lora
    ],
    task_type=TaskType.SEQ_CLS,  # 序列分类（奖励模型就是分类/回归任务）
    lora_alpha=8,
    lora_dropout=0.05,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

trainable params: 2,200,448 || all params: 496,234,112 || trainable%: 0.4434


rank=8/4 对应的 lora = 16/8

In [14]:
# 加载数据
items = []
with open("C:/Users/hhm18/Desktop/深度学习/env_DRL/data/preference.json", "r", encoding="utf8") as f:
    for line in f:
        item = json.loads(line)
        items.append(item)

dataset = Dataset.from_list(items)
dataset,type(dataset)


(Dataset({
     features: ['question', 'chosen', 'rejected'],
     num_rows: 10
 }),
 datasets.arrow_dataset.Dataset)

In [15]:
def process_func(example):
    chosen = example["question"] + example["chosen"]
    rejected = example["question"] + example["rejected"]

    tokenized_chosen = tokenizer(chosen)
    tokenized_rejected = tokenizer(rejected)

    new_example = {}
    new_example["input_ids_chosen"] = tokenized_chosen["input_ids"]
    new_example["attention_mask_chosen"] = tokenized_chosen["attention_mask"]
    new_example["input_ids_rejected"] = tokenized_rejected["input_ids"]
    new_example["attention_mask_rejected"] = tokenized_rejected["attention_mask"]
    return new_example


dataset_map = dataset.map(process_func, remove_columns=['question', 'chosen', 'rejected'])
print(dataset_map, type(dataset_map))


Map: 100%|█████████████████████████████████████████████████████████████████████| 10/10 [00:00<00:00, 805.34 examples/s]

Dataset({
    features: ['input_ids_chosen', 'attention_mask_chosen', 'input_ids_rejected', 'attention_mask_rejected'],
    num_rows: 10
}) <class 'datasets.arrow_dataset.Dataset'>


In [16]:
print(dataset[0], "\n", dataset_map[0])

{'question': 'Python中的字典是什么？', 'chosen': 'Python中的字典是一种无序的可变容器，允许使用键-值对来存储数据。', 'rejected': 'Python中的字典用于存储数据。'} 
 {'input_ids_chosen': [30280, 101047, 18600, 99548, 102021, 11319, 30280, 101047, 18600, 99548, 101158, 42192, 32044, 9370, 30440, 74040, 109300, 3837, 102496, 37029, 60949, 12, 25511, 32664, 36407, 105653, 20074, 1773], 'attention_mask_chosen': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'input_ids_rejected': [30280, 101047, 18600, 99548, 102021, 11319, 30280, 101047, 18600, 99548, 100751, 105653, 20074, 1773], 'attention_mask_rejected': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


- 处理后的数据集包含:
  - `chosen` 和 `rejected` 两个版本的输入。

- 训练时，模型会学到：
  - **chosen 的得分要高于 rejected**。

In [17]:
# 训练reward model
config = RewardConfig(output_dir="C:/Users/hhm18/Desktop/深度学习/env_DRL/model/reward_model")
config.num_train_epochs = 1
config.per_device_train_batch_size = 1

# 为了兼容 RewardTrainer 新 API，打补丁
if not hasattr(model, "warnings_issued"):
    model.warnings_issued = {}

trainer = RewardTrainer(
    model=model,
    args=config,
    train_dataset=dataset,
    processing_class=tokenizer,  # tokenizer，用于内部构造 RewardDataCollatorWithPadding
)


Filter: 100%|███████████████████████████████████████████████████████████████████████████| 10/10 [00:00<?, ? examples/s]


In [ ]:
trainer.train()
trainer.save_model("C:/Users/hhm18/Desktop/深度学习/env_DRL/model/reward_model")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
You're using a Qwen2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss
10,1.003300


C:\Users\hhm18\miniconda3\envs\env_DRL\lib\site-packages\peft\utils\other.py:1228: UserWarning: Unable to fetch remote file due to the following error (MaxRetryError("HTTPSConnectionPool(host='huggingface.co', port=443): Max retries exceeded with url: /Qwen/Qwen2.5-0.5B-Instruct/resolve/main/config.json (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x00000275E5A95D90>, 'Connection to huggingface.co timed out. (connect timeout=10)'))"), '(Request ID: 6dd78b7c-d9ba-457c-98ac-0e9a31cc81df)') - silently ignoring the lookup for the file config.json in Qwen/Qwen2.5-0.5B-Instruct.
  warnings.warn(
C:\Users\hhm18\miniconda3\envs\env_DRL\lib\site-packages\peft\utils\save_and_load.py:286: UserWarning: Could not find a config file in Qwen/Qwen2.5-0.5B-Instruct - will assume that the vocabulary was not modified.
  warnings.warn(
C:\Users\hhm18\miniconda3\envs\env_DRL\lib\site-packages\peft\utils\other.py:1228: UserWarning: Unable to fetch remote file due to the follo

# PPO Training

In [19]:
import torch
from datasets import Dataset
import json

In [20]:
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoTokenizer, BitsAndBytesConfig, AutoModelForSequenceClassification
from trl import RewardConfig, RewardTrainer, AutoModelForCausalLMWithValueHead
from trl import PPOConfig, PPOTrainer

In [21]:
import transformers
import bitsandbytes as bnb 
print("Transformers version:", transformers.__version__)
print("bitsandbytes version:", bnb.__version__)

Transformers version: 4.56.0
bitsandbytes version: 0.47.0


In [22]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,                    # 需要pip bnb
    bnb_4bit_use_double_quant=True,       
    bnb_4bit_compute_dtype=torch.float16,
)

# 设置lora超参，训练奖励模型
peft_config = LoraConfig(
    r = 4,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",             # 对投影层lora
    ],
    task_type=TaskType.SEQ_CLS,  # 序列分类（奖励模型就是分类/回归任务）
    lora_alpha=8,
    lora_dropout=0.05,
)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [23]:
# 套用之前的量化，Policy/Value-model
model_name = "Qwen/Qwen2.5-0.5B-Instruct"
model_path = r"C:/Users/hhm18/Desktop/深度学习/env_DRL/model/reward_model"
Model = AutoModelForCausalLMWithValueHead.from_pretrained(
    model_name,
    reward_adapter=model_path,
    peft_config=peft_config,
    quantization_config=bnb_config,
)
Model.to(device)

AutoModelForCausalLMWithValueHead(
  (pretrained_model): PeftModelForSequenceClassification(
    (base_model): LoraModel(
      (model): Qwen2ForCausalLM(
        (model): Qwen2Model(
          (embed_tokens): Embedding(151936, 896)
          (layers): ModuleList(
            (0-23): 24 x Qwen2DecoderLayer(
              (self_attn): Qwen2Attention(
                (q_proj): lora.Linear4bit(
                  (base_layer): Linear4bit(in_features=896, out_features=896, bias=True)
                  (lora_dropout): ModuleDict(
                    (default): Dropout(p=0.05, inplace=False)
                    (reward_adapter): Dropout(p=0.05, inplace=False)
                  )
                  (lora_A): ModuleDict(
                    (default): Linear(in_features=896, out_features=4, bias=False)
                    (reward_adapter): Linear(in_features=896, out_features=4, bias=False)
                  )
                  (lora_B): ModuleDict(
                    (default): Linear(in_featu

In [24]:
# ref_model(约束KL div) 复制权重
import copy

ref_model = copy.deepcopy(Model)
ref_model.requires_grad_(False)

AutoModelForCausalLMWithValueHead(
  (pretrained_model): PeftModelForSequenceClassification(
    (base_model): LoraModel(
      (model): Qwen2ForCausalLM(
        (model): Qwen2Model(
          (embed_tokens): Embedding(151936, 896)
          (layers): ModuleList(
            (0-23): 24 x Qwen2DecoderLayer(
              (self_attn): Qwen2Attention(
                (q_proj): lora.Linear4bit(
                  (base_layer): Linear4bit(in_features=896, out_features=896, bias=True)
                  (lora_dropout): ModuleDict(
                    (default): Dropout(p=0.05, inplace=False)
                    (reward_adapter): Dropout(p=0.05, inplace=False)
                  )
                  (lora_A): ModuleDict(
                    (default): Linear(in_features=896, out_features=4, bias=False)
                    (reward_adapter): Linear(in_features=896, out_features=4, bias=False)
                  )
                  (lora_B): ModuleDict(
                    (default): Linear(in_featu

In [41]:
# 需要显示指定num_labels使得模型对齐
RM_config = RewardConfig(model_path)
RM_config.num_labels = 1   # 强制设置成1

In [43]:
reward_model = AutoModelForSequenceClassification.from_pretrained(
                                                        model_path, 
                                                        config=RM_config)
reward_model.to(device)

Some weights of Qwen2ForSequenceClassification were not initialized from the model checkpoint at Qwen/Qwen2.5-0.5B-Instruct and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


RuntimeError: Error(s) in loading state_dict for Qwen2ForSequenceClassification:
	size mismatch for score.modules_to_save.default.weight: copying a param with shape torch.Size([1, 896]) from checkpoint, the shape in current model is torch.Size([2, 896]).

In [25]:
items = []
with open("C:/Users/hhm18/Desktop/深度学习/env_DRL/data/queries.json", "r", encoding="utf8") as f:
    for line in f:
        items.append(json.loads(line))
queries_dataset = Dataset.from_list(items)

In [26]:
type(queries_dataset), queries_dataset[0]

(datasets.arrow_dataset.Dataset, {'query': '请给出保持健康的三个方法。'})

In [27]:

def collator(data):
    queries = []
    for item in data:
        queries.append(tokenizer(item["query"],    # 保证模型理解这是个query
                                 return_tensors="pt")["input_ids"].squeeze(
                                        0            # 这里squeeze是去掉自动添加的batch_size
                                        ).to("cuda"))# shape: (1, seq_len) -> (s,)
    return queries                                          

In [29]:
tokenizer = AutoTokenizer.from_pretrained(
            model_name, padding=True, return_tensors="pt")

# 训练设置 clip默认0.2
ppo_config = PPOConfig(
    kl_estimator="K1", 
    num_ppo_epochs=4, 
    batch_size=2, 
    mini_batch_size=1)


In [32]:
generation_kwargs = {
    "min_length": -1,
    "top_k": 0.0,
    "top_p": 1.0,
    "do_sample": True,
    "pad_token_id": tokenizer.pad_token_id,
    "max_new_tokens": 32,
}

In [33]:
print(Model.score.weight.shape)

torch.Size([1, 896])


In [36]:
print(ref_model.score.weight.shape)

torch.Size([1, 896])


In [ ]:
ppo_trainer = PPOTrainer(
    args=ppo_config, 
    model=Model,
    # value_model=Model,
    reward_model=reward_model,
    ref_model=ref_model, 
    train_dataset=queries_dataset, 
    data_collator=collator,         # 还支持多模态任务
    processing_class = tokenizer)



Some weights of Qwen2ForSequenceClassification were not initialized from the model checkpoint at Qwen/Qwen2.5-0.5B-Instruct and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


RuntimeError: Error(s) in loading state_dict for Qwen2ForSequenceClassification:
	size mismatch for score.modules_to_save.default.weight: copying a param with shape torch.Size([1, 896]) from checkpoint, the shape in current model is torch.Size([2, 896]).

[Approximating KL Divergence](http://joschu.net/blog/kl-approx.html)

In [ ]:
for batch in ppo_trainer.dataloader:
    query_tensors = batch

    response_tensors = ppo_trainer.generate(
        query_tensors, return_prompt=False,  **generation_kwargs)
    scores = []
    for query, response in zip(query_tensors, response_tensors):
        input_ids = torch.concat([query, response], dim=0)
        input_ids = torch.unsqueeze(input_ids, dim=0)
        score = ppo_trainer.model.compute_reward_score(input_ids=input_ids)[0, -1, 0]
        scores.append(score)
    stats = ppo_trainer.step(query_tensors, response_tensors, scores)
ppo_trainer.save_pretrained("./rl_model")

## 附录 PPO中的四个模型及其职责

PPO阶段需要四个模型协同工作，以解决强化学习训练中的核心挑战：明确优化方向、稳定训练过程、防止策略遗忘与崩溃。

| **模型名称** | **角色与职责** | **训练过程中是否更新** |
| :--- | :--- | :--- |
| **策略模型 (Actor)** | 这是我们要**主要训练的目标模型**。它接收指令（Prompt），并生成文本（Response）。其参数通过PPO算法不断更新，以生成能获得更高奖励的回答。 | **是** |
| **奖励模型 (Judge)** | 第二阶段训练好的模型，**参数固定**。它负责评估策略模型生成的"指令-回答"对的质量，并给出一个标量奖励分数（Reward）。这是策略模型优化的核心依据。 | 否 |
| **价值模型 (Critic)** | **"评论家"**，其作用是评估在给定指令（状态）下，策略模型可能获得的**未来总奖励的期望值（Value）**。通过预测当前策略的好坏，帮助减少奖励信号的方差，稳定训练过程，提高效率。Critic模型也是一个神经网络，**其参数会随训练更新**。 | **是** |
| **引用模型 (Anchor)** | 通常是**初始的SFT模型的一个副本**，**参数固定**。它作为"锚点"或"基线"，通过计算其与策略模型输出概率分布的KL散度，产生一个惩罚项，防止策略模型在追求高奖励的过程中"走火入魔"，过度偏离其原始的语言能力和知识。 | 否 |